# Cuantización: cómo cabe un modelo gigante en tu portátil

**Facsímil 3 · Arquitecturas y modelos** — capítulos 6 y 7
(modelos abiertos; inferencia optimizada, *edge AI* y hardware).

Un modelo de 7.000 millones de parámetros en *float32* ocupa unos 28 GB solo de pesos: no cabe en una
tarjeta gráfica modesta ni en tu portátil. Y sin embargo hoy corres modelos así en casa. ¿El truco?
La **cuantización**: guardar cada peso con menos bits. En este cuaderno la haces a mano, mides las dos
caras del trato —cuánta **memoria** ahorras y cuánto **error** introduces— y descubres por qué un
formato como `Q4` (los que ves en los modelos GGUF) vive justo en el filo de lo aceptable. No es magia:
es redondear con cabeza.

### Qué vas a aprender
- Por qué los modelos ocupan tanto y qué significa guardar un peso en *float32* frente a *int8*.
- El proceso de cuantización: encontrar la **escala**, **redondear** a la rejilla, y volver.
- A medir el **error** que introduce y por qué casi siempre «cuela».
- Qué pasa con **menos bits** (4, 2) y con **valores atípicos**, los dos enemigos de la cuantización.
- Por qué cuantizar **por filas** mejora la precisión.

### Cuánto cuesta
Unos 10 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import numpy as np
np.random.seed(0)

# Simulamos una capa: una matriz de pesos en float32 (lo normal en un modelo).
pesos = np.random.randn(1024, 1024).astype(np.float32)
print("Matriz de pesos:", pesos.shape, "| tipo:", pesos.dtype)
print(f"Memoria en float32: {pesos.nbytes/1e6:.2f} MB  (4 bytes por numero)")
print(f"Un modelo de 7.000 millones de pesos en float32 ocuparia ~{7e9*4/1e9:.0f} GB. Por eso importa.")


Matriz de pesos: (1024, 1024) | tipo: float32
Memoria en float32: 4.19 MB  (4 bytes por numero)
Un modelo de 7.000 millones de pesos en float32 ocuparia ~28 GB. Por eso importa.


## 1. Cuantizar a 8 bits: redondear a una rejilla

En *float32* cada número usa 32 bits y puede tomar casi cualquier valor. En *int8* solo hay **256
valores** posibles (de -128 a 127). La idea de la cuantización:

1. Encontrar el peso de **mayor magnitud** (define el rango).
2. Dividir ese rango en 256 escalones: la **escala** = (máximo) / 127.
3. Guardar cada peso como el **escalón entero más cercano** (esto es lo que ocupa solo 8 bits).
4. Para *usar* el peso, multiplicarlo de vuelta por la escala (*descuantizar*).

Es un redondeo, ida y vuelta. Lo que se guarda son los enteros y un único número (la escala) por
matriz.


In [2]:
def cuantiza(w, bits=8):
    nivel = 2**(bits-1) - 1                 # 127 para 8 bits
    escala = np.abs(w).max() / nivel        # un solo numero por matriz
    q = np.round(w / escala).astype(np.int64)
    q = np.clip(q, -nivel, nivel)
    return q, escala

q, escala = cuantiza(pesos, bits=8)
recuperado = q.astype(np.float32) * escala

print(f"Pesos en 8 bits -> memoria: {q.astype(np.int8).nbytes/1e6:.2f} MB")
print(f"Reduccion: {pesos.nbytes / q.astype(np.int8).nbytes:.0f} veces mas pequeno (de 32 a 8 bits)\n")
print("Ejemplo (5 pesos):")
print("  original :", np.round(pesos[0,:5], 4))
print("  recuperado:", np.round(recuperado[0,:5], 4))


Pesos en 8 bits -> memoria: 1.05 MB
Reduccion: 4 veces mas pequeno (de 32 a 8 bits)

Ejemplo (5 pesos):
  original : [1.7641 0.4002 0.9787 2.2409 1.8676]
  recuperado: [1.7725 0.3939 0.9847 2.2451 1.8512]


## 2. ¿Cuánto error metimos?

Comparamos los pesos originales con los recuperados. El error existe (hemos redondeado), pero conviene
ponerle número y compararlo con lo que ahorramos. Una medida útil es la **relación señal/ruido** (SNR)
en decibelios: cuánta «señal» (el peso) hay frente al «ruido» (el error de redondeo). Cuanto más alta,
mejor.


In [3]:
def snr_db(original, recuperado):
    senal = np.sqrt((original**2).mean())
    ruido = np.sqrt(((original - recuperado)**2).mean())
    return 20*np.log10(senal/ruido)

error = np.abs(pesos - recuperado)
print(f"Error medio por peso:  {error.mean():.6f}")
print(f"Error maximo por peso: {error.max():.6f}")
print(f"Senal/ruido (8 bits):  {snr_db(pesos, recuperado):.1f} dB (cuanto mas alto, mejor)")
print("\nResumen del trato: 4x menos memoria a cambio de un error minusculo en cada peso.")


Error medio por peso:  0.009854
Error maximo por peso: 0.019694
Senal/ruido (8 bits):  38.9 dB (cuanto mas alto, mejor)

Resumen del trato: 4x menos memoria a cambio de un error minusculo en cada peso.


## 3. Menos bits, más ahorro... y más error

8 bits no es el límite. Los formatos modernos bajan a 4 bits (`Q4`) o incluso menos, ahorrando aún más
memoria. Pero el error crece: con menos escalones, el redondeo es más brusco. Medimos el trato a 8, 4 y
2 bits para ver dónde deja de compensar.


In [4]:
print("bits | valores posibles | memoria relativa | señal/ruido")
for bits in [8, 4, 2]:
    qb, esb = cuantiza(pesos, bits=bits)
    rec = qb.astype(np.float32) * esb
    print(f"  {bits}  |       {2**bits:>4}        |      {bits/32*100:>3.0f}%       |   {snr_db(pesos, rec):>5.1f} dB")
print("\nDe 8 a 4 bits: la mitad de memoria, pero el error sube. A 2 bits ya se degrada mucho.")
print("Ese filo (4 bits) es justo donde viven los modelos GGUF 'Q4' que ves para correr en casa.")


bits | valores posibles | memoria relativa | señal/ruido
  8  |        256        |       25%       |    38.9 dB
  4  |         16        |       12%       |    13.7 dB
  2  |          4        |        6%       |     0.2 dB

De 8 a 4 bits: la mitad de memoria, pero el error sube. A 2 bits ya se degrada mucho.
Ese filo (4 bits) es justo donde viven los modelos GGUF 'Q4' que ves para correr en casa.


## 4. El enemigo nº1: los valores atípicos

La cuantización tiene un talón de Aquiles. La escala la fija el peso de **mayor magnitud**. Si un único
peso es un *outlier* enorme, estira la escala para todos: los escalones se vuelven gruesos y *todos* los
demás pesos (los normales) se cuantizan peor. Lo provocamos metiendo un solo valor atípico y vemos cómo
empeora el error del resto.


In [5]:
pesos_out = pesos.copy()
pesos_out[0, 0] = 60.0          # un solo outlier gigante
q1, e1 = cuantiza(pesos, bits=8);        rec1 = q1.astype(np.float32)*e1
q2, e2 = cuantiza(pesos_out, bits=8);    rec2 = q2.astype(np.float32)*e2
# error solo en los pesos NORMALES (excluyendo el outlier)
err_sin = np.abs(pesos[1:] - rec1[1:]).mean()
err_con = np.abs(pesos_out[1:] - rec2[1:]).mean()
print(f"Error medio de los pesos normales SIN outlier: {err_sin:.6f}")
print(f"Error medio de los pesos normales CON outlier: {err_con:.6f}")
print(f"\nUn solo peso atipico ha empeorado el error de TODOS los demas x{err_con/err_sin:.1f}.")
print("Por eso las tecnicas buenas tratan los outliers aparte.")


Error medio de los pesos normales SIN outlier: 0.009854
Error medio de los pesos normales CON outlier: 0.118132

Un solo peso atipico ha empeorado el error de TODOS los demas x12.0.
Por eso las tecnicas buenas tratan los outliers aparte.


## 5. La defensa: cuantizar por filas

Una mejora barata: en vez de una escala única para toda la matriz, usar **una escala por fila**. Así un
*outlier* en una fila solo afecta a esa fila, no a las demás. Cuesta un poco más de memoria (varias
escalas en vez de una), pero reduce el error. Lo comprobamos sobre la matriz con *outlier*.


In [6]:
def cuantiza_por_filas(w, bits=8):
    nivel = 2**(bits-1) - 1
    escala = np.abs(w).max(axis=1, keepdims=True) / nivel     # una escala por fila
    q = np.clip(np.round(w / escala), -nivel, nivel)
    return q * escala

rec_global = cuantiza(pesos_out, 8)[0].astype(np.float32) * cuantiza(pesos_out, 8)[1]
rec_filas  = cuantiza_por_filas(pesos_out, 8)
print(f"Error medio (escala global): {np.abs(pesos_out - rec_global).mean():.6f}")
print(f"Error medio (escala por fila): {np.abs(pesos_out - rec_filas).mean():.6f}")
print("\nAislar el outlier en su fila protege al resto. Mas escalas, mas precision, algo mas de memoria.")


Error medio (escala global): 0.118130
Error medio (escala por fila): 0.006878

Aislar el outlier en su fila protege al resto. Mas escalas, mas precision, algo mas de memoria.


## 6. ¿Se nota al multiplicar? El efecto compuesto

Un modelo no usa los pesos sueltos: los multiplica por las entradas, miles de veces. Veamos si el error
de cuantización se acumula y estropea el resultado de una operación típica (multiplicar la matriz por un
vector), que es lo que hace una capa.


In [7]:
x = np.random.randn(1024).astype(np.float32)
y_real = pesos @ x
y_cuant = (cuantiza(pesos, 8)[0].astype(np.float32) * cuantiza(pesos, 8)[1]) @ x
dif = np.abs(y_real - y_cuant)
print(f"Salida real vs cuantizada (8 bits) -> error medio {dif.mean():.4f} sobre valores de ~{np.abs(y_real).mean():.2f}")
print(f"Error relativo: {100*dif.mean()/np.abs(y_real).mean():.2f}%")
print("El error existe pero es pequeno frente a la senal: por eso la cuantizacion suele 'colar'.")


Salida real vs cuantizada (8 bits) -> error medio 0.2937 sobre valores de ~25.90
Error relativo: 1.13%
El error existe pero es pequeno frente a la senal: por eso la cuantizacion suele 'colar'.


## 7. Pruébalo tú

1. **Cuantiza a 3 bits** y mira el error relativo de la operación. ¿En qué punto el resultado empieza a
   ser inservible? Ahí está el límite práctico.
2. **Varios *outliers*:** mete 10 valores atípicos. ¿Empeora más la escala global? ¿Aguanta mejor la
   escala por filas?
3. **Cuantización asimétrica:** los pesos no siempre están centrados en 0. Investiga cómo una escala +
   un *desplazamiento* (zero-point) aprovecha mejor los 256 niveles.
4. **Mide la memoria de un modelo real:** ¿cuánto ocuparía uno de 13.000 millones de pesos en 4 bits
   frente a 16 bits? ¿Cabe en tu tarjeta?


## 8. Errores comunes

- **Cuantizar sin mirar los *outliers*.** Un solo peso enorme degrada todos los demás. Las técnicas
  serias los detectan y tratan aparte.
- **Bajar demasiado los bits.** A 2 bits casi todo se rompe; 4 bits es el filo habitual. No es gratis.
- **Olvidar que la escala importa.** Una escala por matriz es lo más barato pero lo menos preciso; por
  fila (o por bloque) es el estándar práctico.
- **Creer que cuantizar es siempre seguro.** En la mayoría de casos «cuela», pero en tareas muy
  sensibles puede degradar la calidad de forma notable: hay que medirlo, no asumirlo.


## 9. Qué te llevas

- **Cuantizar es redondear** los pesos a una rejilla con menos bits: 4× menos memoria al pasar de 32 a 8
  bits (y más aún a 4), con un error que casi siempre «cuela».
- El error **no es gratis**: con muy pocos bits o con *outliers* puede degradar el modelo. De ahí los
  trucos del oficio (por fila, *outliers* aparte, formatos como `Q4`).
- Es la razón técnica de que modelos enormes corran en hardware modesto (*edge AI*, capítulo 7): no se
  «encoge» el modelo, se **representa** más barato.

**Para seguir:** el facsímil 4 te hace elegir entre nube y local sopesando precisamente esto —memoria,
latencia, coste y calidad—, y verás los formatos cuantizados (GGUF, GPTQ, AWQ) en acción.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*